<span style="font-size:2em; font-weight:bold;">Objective 2: Evaluation of Ground Motion Spatial Correlation in New Zealand Through Empirical Approaches</span>
***

In this notebook we compute the ground-motion within event residuals and fit a stationary SCM.

# 1.0 Import Modules and Data

## 1.1 Import Modules

In [ ]:
# Repository paths (works when Jupyter starts at the repository root or in notebooks/).
from pathlib import Path
import sys

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (parent for parent in (_current, *_current.parents)
     if (parent / "src" / "cbb21_cossim.py").is_file()
     and (parent / "data").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Repository root not found. Start Jupyter in the repository or notebooks/ directory.")

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = REPO_ROOT / "data"
RESIDUALS_DIR = DATA_DIR / "residuals"
SCM_DIR = DATA_DIR / "nzwide_scm"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# need to disable short-vector for openquake
import os
os.environ["NUMBA_DISABLE_INTEL_SVML"] = "1"

import importlib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
from matplotlib.collections import LineCollection
import seaborn as sns
from pyproj import Geod
import geopandas as gpd
from scipy.optimize import minimize
from scipy.stats import invgamma
import json
import shapely
from shapely.geometry import Point, Polygon, shape
from openquake.hazardlib import const, imt, gsim
from openquake.hazardlib.site import Site, SiteCollection
from openquake.hazardlib.geo import Point

# For OpenQuake GMPE logic
stddev_types = [
    const.StdDev.TOTAL,
    const.StdDev.INTER_EVENT,
    const.StdDev.INTRA_EVENT
]

## 1.2 Define Functions

In [ ]:
# compute azimuth and epicentral distance
def LongLatToPolar(Xst, Xeq):
    geod = Geod(ellps='WGS84')
    r = geod.inv(lons1 = Xeq[:, 0], lats1 = Xeq[:, 1],
        lons2 = Xst[:, 0], lats2 = Xst[:, 1])
    repi = r[2]/1000
    az = r[0]*np.pi/180
    az[az<0] += 2*np.pi
    Xp = np.vstack([repi, az]).T
    return Xp

# get backarc stations
def get_backarc_mask(backarc_json_ffp, locs):
    with open(backarc_json_ffp) as f:
        gj = json.load(f)

    geom = shape(gj["geometry"])

    # Build a vectorized array of Points
    pts = shapely.points(locs[:, 0], locs[:, 1])

    # Vectorized contains (no Python loop, no GEOS mismatch)
    return shapely.contains(geom, pts)

# compute within-event residuals
def ab92_fit_one_period(im_obs_col, im_sim_col, evid_col, SA_tau_col, SA_phi_col, *, center=False, tol=1e-10, maxiter=200):
    mask = np.isfinite(im_obs_col) & np.isfinite(im_sim_col)
    y  = np.asarray(im_obs_col[mask], float)
    mu = np.asarray(im_sim_col[mask], float)
    phi_gmm = np.asarray(SA_phi_col[mask], float)
    tau_gmm = np.asarray(SA_tau_col[mask], float)
    ev = pd.Series(np.asarray(evid_col[mask]))

    Delta = np.log(y) - np.log(mu)
    if center:
        Delta = Delta - Delta.mean()

    df = pd.DataFrame({"evid": ev, "r": Delta})
    phi_gmm_s = pd.Series(phi_gmm, index=df.index)
    tau_gmm_s = pd.Series(tau_gmm, index=df.index)
    grp = df.groupby("evid")

    n      = grp.size()                                      # n_i
    sum_r  = grp["r"].sum()                                  # ΣΔ_ij
    sum_r2 = grp["r"].apply(lambda x: float(np.square(x).sum()))
    N, M = int(n.sum()), len(n)

    # seed initial values based on GMM phi/tau
    phi2_0 = float(np.mean(phi_gmm)**2)
    tau2_0 = float(np.mean(tau_gmm)**2)

    # NEGATIVE log-likelihood (FIXED quadratic term)
    def nll(log_phi2, log_tau2):
        phi2 = np.exp(log_phi2); tau2 = np.exp(log_tau2)
        ld = ((n - 1) * np.log(phi2) + np.log(phi2 + n * tau2)).sum()
        quad = ( (sum_r2 / phi2) - (tau2 / (phi2 * (phi2 + n * tau2))) * (sum_r ** 2) ).sum()
        return 0.5 * (ld + quad) + 0.5 * N * np.log(2 * np.pi)

    res = minimize(lambda x: nll(x[0], x[1]),
                   x0=np.log([phi2_0, tau2_0]),
                   method="L-BFGS-B",
                   bounds=[(np.log(1e-12), np.log(1e6)), (np.log(1e-12), np.log(1e6))],
                   options=dict(maxiter=maxiter, ftol=tol))

    # fallback if needed
    if (not res.success):
        phi2_hat, tau2_hat = phi2_0, tau2_0
    else:
        phi2_hat, tau2_hat = np.exp(res.x[0]), np.exp(res.x[1])

    # tiny floors to avoid blow-ups
    phi2_hat = float(max(phi2_hat, 1e-10))
    tau2_hat = float(max(tau2_hat, 1e-12))
    phi_hat, tau_hat = float(np.sqrt(phi2_hat)), float(np.sqrt(tau2_hat))

    # AY92 δB_i and partition
    deltaB_e = (tau2_hat * sum_r) / (n * tau2_hat + phi2_hat)  # event-level
    deltaB_row = df["evid"].map(deltaB_e)

    eps    = df["r"] - deltaB_row
    dbebar = deltaB_row / tau_hat
    dwbar  = eps / phi_hat

    # optional strict per-event z (mean 0, sd 1) for geostatistics
    z_strict = eps.groupby(df["evid"]).transform(
        lambda s: (s - s.mean()) / (s.std(ddof=1) if s.std(ddof=1) > 0 else np.nan)
    )

    out = {
        "tau": tau_hat, "phi": phi_hat, "converged": res.success, "nll": float(res.fun),
        "dbe_bar": pd.Series(np.nan, index=np.arange(len(im_obs_col))),
        "dw_bar":  pd.Series(np.nan, index=np.arange(len(im_obs_col))),
        "deltaB":  pd.Series(np.nan, index=np.arange(len(im_obs_col))),
        "eps":     pd.Series(np.nan, index=np.arange(len(im_obs_col))),
        "z_strict":pd.Series(np.nan, index=np.arange(len(im_obs_col)))
    }
    out["dbe_bar"].iloc[mask] = dbebar.values
    out["dw_bar"].iloc[mask]  = dwbar.values
    out["deltaB"].iloc[mask]  = deltaB_row.values
    out["eps"].iloc[mask]     = eps.values
    out["z_strict"].iloc[mask]= z_strict.values
    return out


## 1.3 Import Data

### 1.3.1 Create Main Dataframe

In [ ]:
## Load NZ GMDB Flatfile
gm_im_table_50 = pd.read_csv(RESIDUALS_DIR / "ground_motion_im_table_rotd50_flat.csv", low_memory=False, dtype={"sta": "string"})

## Remove non-IM information to optimize storage
gm_im_table_50.drop(gm_im_table_50.iloc[:,182:571],inplace=True,axis = 1)

## Remove Duplicates BUT with HN channels as priority
gm_im_table_50.sort_values('chan',ascending=False,inplace=True)
gm_im_table_50.drop_duplicates(subset=['evid','sta'],keep='first',inplace=True)
gm_im_table_50.sort_values('datetime',ascending=True,inplace=True)

## Remove possible error values
gm_im_table_50 = gm_im_table_50[gm_im_table_50["pSA_0.01"]>0]

## append additional station parameters
stations = pd.read_csv(RESIDUALS_DIR / "site_table_TRI_multiscale_group_TWI.csv", dtype={"sta": "string"})
stations.fillna(0,inplace=True)
stations_filt = pd.DataFrame()
stations_filt['sta'] = stations['sta']
stations_filt['TRI'] = stations['TRI']
stations_filt['TWI'] = stations['TWI']
stations_filt['geoage'] = stations['geoMA_min']
stations_filt['k0'] = stations['k0_mean']
stations_filt['precip'] = stations['precip']
stations_filt['group'] = stations['cregion_group']
gm_im_table_50 = gm_im_table_50.merge(stations_filt,on='sta',how='left')

# apply maxmimum distance filter
gm_im_table_50 = gm_im_table_50[gm_im_table_50.r_jb<=300].reset_index(drop=True)

## Apply filter on the data: Tectonic Classification
gm_im_table_50_crustal = gm_im_table_50[(gm_im_table_50.tect_class=='Crustal')&(gm_im_table_50.mag>=3.5)]
gm_im_table_50_slab = gm_im_table_50[(gm_im_table_50.tect_class=='Slab')&(gm_im_table_50.mag>=5)]
gm_im_table_50_interface = gm_im_table_50[(gm_im_table_50.tect_class=='Interface')&(gm_im_table_50.mag>=5)]
gm_im_table_50 = pd.concat([gm_im_table_50_crustal, gm_im_table_50_slab, gm_im_table_50_interface],axis=0)
gm_im_table_50 = gm_im_table_50.reset_index(drop = True)

# compute epicentral distance and epicentral azimuth
Xstat = gm_im_table_50[['sta_lon','sta_lat']].values
Xevid = np.zeros_like(Xstat)
Xevid[:,0] = gm_im_table_50.ev_lon
Xevid[:,1] = gm_im_table_50.ev_lat
Xp = LongLatToPolar(Xstat, Xevid)
gm_im_table_50['epi_dist'] = Xp[:,0]
gm_im_table_50['epi_azimuth'] = Xp[:,1]

# get backarc masks
backarc_mask = get_backarc_mask(RESIDUALS_DIR / "backarc.json", gm_im_table_50[["sta_lon", "sta_lat"]].values)
gm_im_table_50['backarc'] = backarc_mask

### 1.3.2 Store maximum useable periods: All data

In [ ]:
# define periods
pSA_T  = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
pSAobs_T = ['PGA', 'pSA_0.03', 'pSA_0.06', 'pSA_0.1','pSA_0.3', 'pSA_0.6', 'pSA_1.0', 'pSA_2.0', 'pSA_3.0', 'pSA_4.0', 'pSA_5.0', 'pSA_6.0', 'pSA_7.0', 'pSA_8.0', 'pSA_9.0', 'pSA_10.0']

# define lists for storage
gm_im_table_50_sa = []
n_rec_sa = []
n_ev_sa = []
n_sta = []
n_rec_crus = []
n_ev_crus = []
n_rec_slab = []
n_ev_slab = []
n_rec_inter = []
n_ev_inter = []

# zip for plot
plot = False
all_classes = sorted(set(gm_im_table_50['tect_class'].dropna().unique()).union(gm_im_table_50['tect_class'].dropna().unique()))
palette = {cls:col for cls,col in zip(all_classes, sns.color_palette('tab10', len(all_classes)))}

for i in tqdm(range(len(pSA_T))):
    # minimum
    gm_im_table_50_temp = gm_im_table_50[gm_im_table_50[['fmin_X','fmin_Y']].min(axis = 1)<=0.8*(1/pSA_T[i])].reset_index(drop=True)

    ## Apply filter on minimum number of records per earthquake
    evid_40 = gm_im_table_50_temp.groupby('evid').count()
    evid_40 = evid_40[evid_40.record_id>=3]
    nevid_40 = evid_40.iloc[:,0]
    evid_40 = evid_40.index.tolist()
    gm_im_table_50_temp = gm_im_table_50_temp[gm_im_table_50_temp.evid.isin(evid_40)].reset_index(drop=True)
    print('number of events: %.0f with %.0f records'%(len(evid_40),len(gm_im_table_50_temp)))

    # append data
    gm_im_table_50_sa.append(gm_im_table_50_temp)
    n_rec_sa.append(len(gm_im_table_50_temp)); n_ev_sa.append(len(evid_40))
    n_sta.append(len(gm_im_table_50_temp["sta"].unique()))

    n_rec_crus.append(len(gm_im_table_50_temp[gm_im_table_50_temp.tect_class=='Crustal'])); n_ev_crus.append(len(gm_im_table_50_temp[gm_im_table_50_temp.tect_class=='Crustal'].groupby('evid').count()))

    n_rec_slab.append(len(gm_im_table_50_temp[gm_im_table_50_temp.tect_class=='Slab'])); n_ev_slab.append(len(gm_im_table_50_temp[gm_im_table_50_temp.tect_class=='Slab'].groupby('evid').count()))

    n_rec_inter.append(len(gm_im_table_50_temp[gm_im_table_50_temp.tect_class=='Interface'])); n_ev_inter.append(len(gm_im_table_50_temp[gm_im_table_50_temp.tect_class=='Interface'].groupby('evid').count()))

    # plot
    if plot:
        p = sns.jointplot(
            data=gm_im_table_50_temp,
            x="r_rup", y="mag",
            hue="tect_class",
            kind="scatter",
            height=7, ratio=4,
            marginal_ticks=True,
            marginal_kws={'multiple': 'stack'},
            palette=palette,
            hue_order=all_classes,
            xlim=(1,1000), ylim=(3.5,8)
        )

        p.ax_joint.set_xscale('log')
        p.fig.tight_layout()
        p.set_axis_labels(xlabel='$R_{rup}$ (km)', ylabel='Magnitude, $M_w$', fontsize=14)

        # Adjust legend and ticks
        plt.legend(title='Tectonic Class', fontsize=12)
        plt.tick_params(axis='both', which='major', labelsize=12)
        p.fig.savefig(f"mag_vs_rrup_{pSAobs_T[i]}.png", dpi=200, bbox_inches="tight")

summary = pd.DataFrame({
    'Period': pSA_T,
    'n_records': n_rec_sa,
    'n_events': n_ev_sa,
    'n_sta': n_sta,
    'n_crustal_records': n_rec_crus,
    'n_crustal_events': n_ev_crus,
    'n_slab_records': n_rec_slab,
    'n_slab_events': n_ev_slab,
    'n_interface_records': n_rec_inter,
    'n_interface_events': n_ev_inter
})

# summary.to_csv(Path(r'C:\Users\LEGION\OneDrive - University of Canterbury\spatial_correlations\BB2023\AJR_NZGMDB_EASZT\data\all_data_summary_n3.csv'))

plt.figure(figsize=(12,6), dpi = 150)
plt.loglog(pSA_T, n_rec_sa, marker='o', label='All Records')
plt.loglog(pSA_T, n_rec_crus, marker='s', label='Crustal Records')
plt.loglog(pSA_T, n_rec_slab, marker='^', label='Slab Records')
plt.loglog(pSA_T, n_rec_inter, marker='d', label='Interface Records')

plt.xlabel('Period, T (s)', fontsize=14)
plt.ylabel('Number of Records, n', fontsize=14)
plt.legend(fontsize=12)
plt.grid(which = 'both')
plt.xlim(0.01,6)
plt.show()

In [ ]:
# define periods
pSA_T  = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
pSAobs_T = ['PGA', 'pSA_0.03', 'pSA_0.06', 'pSA_0.1','pSA_0.3', 'pSA_0.6', 'pSA_1.0', 'pSA_2.0', 'pSA_3.0', 'pSA_4.0', 'pSA_5.0', 'pSA_6.0', 'pSA_7.0', 'pSA_8.0', 'pSA_9.0', 'pSA_10.0']

rows = []

for T in pSA_T:
    if T > 6.0:
        continue

    # same usable-period filter
    df_T = gm_im_table_50[
        gm_im_table_50[["fmin_X", "fmin_Y"]].min(axis=1) <= 0.8 * (1 / T)
    ].reset_index(drop=True)

    # evaluation dataset: n >= 3 records per event
    ev_counts_3 = df_T.groupby("evid")["record_id"].count()
    evid_3 = ev_counts_3[ev_counts_3 >= 3].index
    df_3 = df_T[df_T["evid"].isin(evid_3)].reset_index(drop=True)

    # SCM development dataset: n >= 40 records per event
    ev_counts_40 = df_T.groupby("evid")["record_id"].count()
    evid_40 = ev_counts_40[ev_counts_40 >= 40].index
    df_40 = df_T[df_T["evid"].isin(evid_40)].reset_index(drop=True)

    # sets
    rec_3 = set(df_3["record_id"].astype(str))
    rec_40 = set(df_40["record_id"].astype(str))

    ev_3 = set(df_3["evid"].astype(str))
    ev_40 = set(df_40["evid"].astype(str))

    sta_3 = set(df_3["sta"].astype(str))
    sta_40 = set(df_40["sta"].astype(str))

    # additional evaluation data relative to SCM development data
    add_rec = len(rec_3 - rec_40)
    add_ev = len(ev_3 - ev_40)
    add_sta = len(sta_3 - sta_40)

    rows.append({
        "Period": T,

        "SCM_records": len(rec_40),
        "Additional_records": add_rec,
        "Additional_records_%": 100 * add_rec / len(rec_40),

        "SCM_events": len(ev_40),
        "Additional_events": add_ev,
        "Additional_events_%": 100 * add_ev / len(ev_40),

        "SCM_stations": len(sta_40),
        "Additional_stations": add_sta,
        "Additional_stations_%": 100 * add_sta / len(sta_40),
    })

summary_additional = pd.DataFrame(rows)

# collapse periods <= 0.30 s if identical
cols_to_check = [
    "Additional_records", "Additional_records_%",
    "Additional_events", "Additional_events_%",
    "Additional_stations", "Additional_stations_%"
]

leq_030 = summary_additional[summary_additional["Period"] <= 0.30].copy()
gt_030 = summary_additional[summary_additional["Period"] > 0.30].copy()

if (leq_030[cols_to_check].round(6).nunique() == 1).all():
    first = leq_030.iloc[[0]].copy()
    first["Period_label"] = r"$\leq 0.30$"
    gt_030["Period_label"] = gt_030["Period"].map(lambda x: f"{x:.2f}")
    table_df = pd.concat([first, gt_030], ignore_index=True)
else:
    summary_additional["Period_label"] = summary_additional["Period"].map(lambda x: f"{x:.2f}")
    table_df = summary_additional.copy()

# rounded table values for manuscript
table_df_out = table_df[[
    "Period_label",
    "Additional_records", "Additional_records_%",
    "Additional_events", "Additional_events_%",
    "Additional_stations", "Additional_stations_%"
]].copy()

for col in ["Additional_records_%", "Additional_events_%", "Additional_stations_%"]:
    table_df_out[col] = table_df_out[col].round(0).astype(int)

# print LaTeX rows
for _, r in table_df_out.iterrows():
    print(
        f"{r['Period_label']} & "
        f"{int(r['Additional_records'])} & {int(r['Additional_records_%'])}\\%  & "
        f"{int(r['Additional_events'])} & {int(r['Additional_events_%'])}\\% & "
        f"{int(r['Additional_stations'])} & {int(r['Additional_stations_%'])}\\% \\\\"
    )

### 1.3.3 Store maximum useable periods: Regional Groups

In [ ]:
group1 = ['Group 1: Wellington']
group2 = ['Group 2: Canterbury']
north_group = ['Group 3: West North Island', 'Group 4: East North Island']
south_group = ['Group 5: North South Island', 'Group 6: South South Island']

gm_im_table_50_grp1 = []
gm_im_table_50_grp2 = []
gm_im_table_50_sgrp = []
gm_im_table_50_ngrp = []

for i in tqdm(range(len(pSA_T))):

    # filter by group 1
    gm_im_table_50_reg = gm_im_table_50_sa[i][gm_im_table_50_sa[i].group.isin(group1)].reset_index(drop = True)
    evid_40 = gm_im_table_50_reg.groupby('evid').count()
    evid_40 = evid_40[evid_40.record_id>=3]
    evid_40 = evid_40.index.tolist()
    gm_im_table_50_reg = gm_im_table_50_reg[gm_im_table_50_reg.evid.isin(evid_40)].reset_index(drop=True)
    gm_im_table_50_grp1.append(gm_im_table_50_reg)

    # filter by group 2
    gm_im_table_50_reg = gm_im_table_50_sa[i][gm_im_table_50_sa[i].group.isin(group2)].reset_index(drop = True)
    evid_40 = gm_im_table_50_reg.groupby('evid').count()
    evid_40 = evid_40[evid_40.record_id>=3]
    evid_40 = evid_40.index.tolist()
    gm_im_table_50_reg = gm_im_table_50_reg[gm_im_table_50_reg.evid.isin(evid_40)].reset_index(drop=True)
    gm_im_table_50_grp2.append(gm_im_table_50_reg)

    # filter by north group
    gm_im_table_50_reg = gm_im_table_50_sa[i][gm_im_table_50_sa[i].group.isin(north_group)].reset_index(drop = True)
    evid_40 = gm_im_table_50_reg.groupby('evid').count()
    evid_40 = evid_40[evid_40.record_id>=3]
    evid_40 = evid_40.index.tolist()
    gm_im_table_50_reg = gm_im_table_50_reg[gm_im_table_50_reg.evid.isin(evid_40)].reset_index(drop=True)
    gm_im_table_50_ngrp.append(gm_im_table_50_reg)

    # filter by south group
    gm_im_table_50_reg = gm_im_table_50_sa[i][gm_im_table_50_sa[i].group.isin(south_group)].reset_index(drop = True)
    evid_40 = gm_im_table_50_reg.groupby('evid').count()
    evid_40 = evid_40[evid_40.record_id>=3]
    evid_40 = evid_40.index.tolist()
    gm_im_table_50_reg = gm_im_table_50_reg[gm_im_table_50_reg.evid.isin(evid_40)].reset_index(drop=True)
    gm_im_table_50_sgrp.append(gm_im_table_50_reg)

# 2.0 Compute IMs and Within-event residuals

In [ ]:
save = False
run_imrescalc = False
outdir = RESIDUALS_DIR
outdir.mkdir(parents=True, exist_ok=True)

## 2.1 OpenQuake: Specify Objects

In [ ]:
# Specify and Compute IMs
class Properties(object):
    """
        Use to create an object that stores values for sites, rup and dists.
    """

    def __init__(self):
            # this allows attaching arbitrary attributes to self later
        pass

location = Point(
        0.0, 0.0, 0.0)  # Create a dummy location as OQ calculation doesn't use a location
oq_site = Site(location)
rup = Properties()
dists = Properties()

# NHSM 2022 Models
# Crustal Only Models
CY_2014 = gsim.chiou_youngs_2014.ChiouYoungs2014()
ASK_2014 = gsim.abrahamson_2014.AbrahamsonEtAl2014()
BSSA_2014 = gsim.boore_2014.BooreEtAl2014()
CB_2014 = gsim.campbell_bozorgnia_2014.CampbellBozorgnia2014()
S_2022 = gsim.nz22.stafford_2022.Stafford2022()

# Subduction Only Models (Interface and Slab)
AB_2015SI = gsim.abrahamson_2015.AbrahamsonEtAl2015SInter()
AB_2015SS = gsim.abrahamson_2015.AbrahamsonEtAl2015SSlab()
AG_2020I = gsim.abrahamson_gulerce_2020.AbrahamsonGulerce2020SInter()
AG_2020S = gsim.abrahamson_gulerce_2020.AbrahamsonGulerce2020SSlab()
K_2020I = gsim.kuehn_2020.KuehnEtAl2020SInter()
K_2020S = gsim.kuehn_2020.KuehnEtAl2020SSlab()
P_2020I = gsim.parker_2020.ParkerEtAl2020SInter()
P_2020S = gsim.parker_2020.ParkerEtAl2020SSlab()

## 2.2 Computation: All Data

In [ ]:
if run_imrescalc:
    for iper in tqdm(range(len(gm_im_table_50_sa)), desc = 'Processing Period: '):
        gm_im_table_50_filt = gm_im_table_50_sa[iper]

        SA_im = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_tau = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_phi = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))

        for i in tqdm(range(len(gm_im_table_50_filt)), desc = 'Computing IMs'):

                location = Point(
                        0.0, 0.0, 0.0
                )  # Create a dummy location as OQ calculation doesn't use a location
                oq_site = Site(location)
                rup = Properties()
                dists = Properties()

                ##set input parameters
                #set source properties
                rup.mag = np.array([gm_im_table_50_filt.mag[i]])
                rup.rake = np.array([gm_im_table_50_filt.rake[i]])
                rup.dip = np.array([gm_im_table_50_filt.dip[i]])
                rup.ztor = np.array([gm_im_table_50_filt.z_tor[i]])
                rup.width = np.array([gm_im_table_50_filt.f_width[i]])
                rup.hypo_depth = np.array([gm_im_table_50_filt.ev_depth[i]])

                #set site properties (ALWAYS CHECK UNITS THE MODEL USES, e.g. m or km)
                oq_site.vs30 = np.array([gm_im_table_50_filt.Vs30[i]])    #this needs to be in m/s
                oq_site.vs30measured = np.array([False])
                oq_site.z1pt0 = np.array([gm_im_table_50_filt['Z1.0'][i]])    #this needs to be in m
                oq_site.z2pt5 = np.array([gm_im_table_50_filt['Z2.5'][i]])
                oq_site.backarc = np.array([gm_im_table_50_filt['backarc'][i]])
                sites = SiteCollection([oq_site])

                #set distance properties (generally in km)
                dists.rhypo = np.array([gm_im_table_50_filt.r_hyp[i]])
                dists.rrup = np.array([gm_im_table_50_filt.r_rup[i]])
                dists.repi = np.array([gm_im_table_50_filt.r_epi[i]])
                dists.rjb = np.array([gm_im_table_50_filt.r_jb[i]])
                dists.rx = np.array([gm_im_table_50_filt.r_x[i]])
                dists.ry0 = np.array([gm_im_table_50_filt.r_y[i]])

                if gm_im_table_50_filt.tect_class[i] == 'Interface':
                    GMPE = K_2020I

                    if iper == 0:
                        im_PGA = imt.PGA()
                        m_PGA, stdev_PGA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_PGA, stddev_types)
                        SA_im[i, 0] = np.exp(m_PGA[0])
                        SA_std[i, 0] = stdev_PGA[0][0]
                        SA_std_tau[i, 0] = stdev_PGA[1][0]
                        SA_std_phi[i, 0] = stdev_PGA[2][0]
                    else:
                        im_SA = imt.SA(period=pSA_T[iper])
                        m_SA, stdev_SA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_SA, stddev_types)
                        SA_im[i, iper] = np.exp(m_SA[0])
                        SA_std[i, iper] = stdev_SA[0][0]
                        SA_std_tau[i, iper] = stdev_SA[1][0]
                        SA_std_phi[i, iper] = stdev_SA[2][0]

                elif gm_im_table_50_filt.tect_class[i] == 'Slab':
                    GMPE = K_2020S

                    if iper == 0:
                        im_PGA = imt.PGA()
                        m_PGA, stdev_PGA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_PGA, stddev_types)
                        SA_im[i, 0] = np.exp(m_PGA[0])
                        SA_std[i, 0] = stdev_PGA[0][0]
                        SA_std_tau[i, 0] = stdev_PGA[1][0]
                        SA_std_phi[i, 0] = stdev_PGA[2][0]
                    else:
                        im_SA = imt.SA(period=pSA_T[iper])
                        m_SA, stdev_SA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_SA, stddev_types)
                        SA_im[i, iper] = np.exp(m_SA[0])
                        SA_std[i, iper] = stdev_SA[0][0]
                        SA_std_tau[i, iper] = stdev_SA[1][0]
                        SA_std_phi[i, iper] = stdev_SA[2][0]

                else:
                    GMPE = CY_2014

                    if iper == 0:
                        im_PGA = imt.PGA()
                        m_PGA, stdev_PGA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_PGA, stddev_types)
                        SA_im[i, 0] = np.exp(m_PGA[0])
                        SA_std[i, 0] = stdev_PGA[0][0]
                        SA_std_tau[i, 0] = stdev_PGA[1][0]
                        SA_std_phi[i, 0] = stdev_PGA[2][0]
                    else:
                        im_SA = imt.SA(period=pSA_T[iper])
                        m_SA, stdev_SA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_SA, stddev_types)
                        SA_im[i, iper] = np.exp(m_SA[0])
                        SA_std[i, iper] = stdev_SA[0][0]
                        SA_std_tau[i, iper] = stdev_SA[1][0]
                        SA_std_phi[i, iper] = stdev_SA[2][0]

        # create im_sim/im_obs dataframe
        im_obs = pd.DataFrame()
        im_obs['evid'] = gm_im_table_50_filt.evid
        im_obs['sta'] = gm_im_table_50_filt.sta
        im_obs['sta_lat'] = gm_im_table_50_filt.sta_lat
        im_obs['sta_lon'] = gm_im_table_50_filt.sta_lon
        im_obs['vs30'] = gm_im_table_50_filt.Vs30
        im_obs['epicenter_dist'] = gm_im_table_50_filt.epi_dist
        im_obs['epicenter_azimuth'] = gm_im_table_50_filt.epi_azimuth
        im_obs['Z1.0'] = gm_im_table_50_filt['Z1.0']
        im_obs['TRI'] = gm_im_table_50_filt['TRI']
        im_obs['TWI'] = gm_im_table_50_filt['TWI']
        im_obs['geoage'] = gm_im_table_50_filt['geoage']
        im_obs['k0'] = gm_im_table_50_filt['k0']
        im_obs['precip'] = gm_im_table_50_filt['precip']
        im_obs[pSAobs_T[iper]] = gm_im_table_50_filt[pSAobs_T[iper]].values

        im_sim = im_obs.copy()
        im_sim[pSAobs_T[iper]] = SA_im[:,iper]

        # compute within-event residuals
        im_dbebar_ab92 = im_sim.copy()
        im_dbe_ab92 = im_sim.copy()
        im_dwbar_ab92 = im_sim.copy()
        im_residual = im_sim.copy()

        for tect_class in ['Crustal','Interface','Slab']:

            # tect class events
            gm_im_table_50_filt_tect_class = gm_im_table_50_filt[gm_im_table_50_filt.tect_class==tect_class]
            idx_tect_class = gm_im_table_50_filt_tect_class.index
            SA_std_tau_tect_class = SA_std_tau[idx_tect_class]
            SA_std_phi_tect_class = SA_std_phi[idx_tect_class]
            im_residual_tect_class = im_residual.iloc[idx_tect_class]
            im_obs_tect_class = im_obs.iloc[idx_tect_class]
            im_sim_tect_class = im_sim.iloc[idx_tect_class]

            period_col = pSAobs_T[iper]   # one period
            res = ab92_fit_one_period(
                    im_obs_tect_class[period_col],           # observed IMs for this period
                    im_sim_tect_class[period_col],           # GMM medians for this period
                    im_residual_tect_class["evid"],          # event IDs
                    SA_std_tau_tect_class[:,iper],
                    SA_std_phi_tect_class[:,iper],
                    center=False                  # set True only if you want to de-bias Δ by its grand mean
            )

            tau_hat, phi_hat = res["tau"], res["phi"]
            im_dbebar_ab92.loc[idx_tect_class, period_col] = res["dbe_bar"].values
            im_dbe_ab92.loc[idx_tect_class, period_col] = res["deltaB"].values
            im_dwbar_ab92.loc[idx_tect_class, period_col]  = res["dw_bar"].values

        if save:
            # save file
            pSA_names = ['PGA','pSA_003','pSA_006','pSA_010','pSA_030','pSA_060','pSA_100','pSA_200','pSA_300','pSA_400','pSA_500','pSA_600','pSA_700','pSA_800','pSA_900','pSA_1000']

            im_dbe_ab92_file = 'NZGMDB_v4p3_dbe_%s'%pSA_names[iper]
            im_dbebar_ab92_file = 'NZGMDB_v4p3_dbebar_%s'%pSA_names[iper]
            im_dwbar_ab92_file = 'NZGMDB_v4p3_dwbar_%s'%pSA_names[iper]

            im_dwbar_ab92.to_csv(outdir / f"{im_dwbar_ab92_file}.csv", index=False)
            im_dbe_ab92.to_csv(outdir / f"{im_dbe_ab92_file}.csv", index=False)
            im_dbebar_ab92.to_csv(outdir / f"{im_dbebar_ab92_file}.csv", index=False)

## 2.3 Computation: Group 1

In [ ]:
if run_imrescalc:
    for iper in tqdm(range(len(gm_im_table_50_grp1)), desc = 'Processing Period: '):
        gm_im_table_50_filt = gm_im_table_50_grp1[iper]

        SA_im = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_tau = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_phi = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))

        for i in tqdm(range(len(gm_im_table_50_filt)), desc = 'Computing IMs'):

                location = Point(
                        0.0, 0.0, 0.0
                )  # Create a dummy location as OQ calculation doesn't use a location
                oq_site = Site(location)
                rup = Properties()
                dists = Properties()

                ##set input parameters
                #set source properties
                rup.mag = np.array([gm_im_table_50_filt.mag[i]])
                rup.rake = np.array([gm_im_table_50_filt.rake[i]])
                rup.dip = np.array([gm_im_table_50_filt.dip[i]])
                rup.ztor = np.array([gm_im_table_50_filt.z_tor[i]])
                rup.width = np.array([gm_im_table_50_filt.f_width[i]])
                rup.hypo_depth = np.array([gm_im_table_50_filt.ev_depth[i]])

                #set site properties (ALWAYS CHECK UNITS THE MODEL USES, e.g. m or km)
                oq_site.vs30 = np.array([gm_im_table_50_filt.Vs30[i]])    #this needs to be in m/s
                oq_site.vs30measured = np.array([False])
                oq_site.z1pt0 = np.array([gm_im_table_50_filt['Z1.0'][i]])    #this needs to be in m
                oq_site.z2pt5 = np.array([gm_im_table_50_filt['Z2.5'][i]])
                oq_site.backarc = np.array([gm_im_table_50_filt['backarc'][i]])
                sites = SiteCollection([oq_site])

                #set distance properties (generally in km)
                dists.rhypo = np.array([gm_im_table_50_filt.r_hyp[i]])
                dists.rrup = np.array([gm_im_table_50_filt.r_rup[i]])
                dists.repi = np.array([gm_im_table_50_filt.r_epi[i]])
                dists.rjb = np.array([gm_im_table_50_filt.r_jb[i]])
                dists.rx = np.array([gm_im_table_50_filt.r_x[i]])
                dists.ry0 = np.array([gm_im_table_50_filt.r_y[i]])

                # choose GMPE
                tect = gm_im_table_50_filt.tect_class.iloc[i]
                if tect == 'Interface':
                    GMPE = K_2020I
                elif tect == 'Slab':
                    GMPE = K_2020S
                else:
                    GMPE = CY_2014

                if iper == 0:
                    imt_obj = imt.PGA()
                else:
                    imt_obj = imt.SA(period=pSA_T[iper])

                m, stdevs = GMPE.get_mean_and_stddevs(SiteCollection([oq_site]), rup, dists, imt_obj, stddev_types)
                SA_im[i,iper]      = np.exp(m[0])
                SA_std[i,iper]     = stdevs[0][0]
                SA_std_tau[i,iper] = stdevs[1][0]
                SA_std_phi[i,iper] = stdevs[2][0]

        # create im_sim/im_obs dataframe
        im_obs = pd.DataFrame()
        im_obs['evid'] = gm_im_table_50_filt.evid
        im_obs['sta'] = gm_im_table_50_filt.sta
        im_obs['sta_lat'] = gm_im_table_50_filt.sta_lat
        im_obs['sta_lon'] = gm_im_table_50_filt.sta_lon
        im_obs['vs30'] = gm_im_table_50_filt.Vs30
        im_obs['epicenter_dist'] = gm_im_table_50_filt.epi_dist
        im_obs['epicenter_azimuth'] = gm_im_table_50_filt.epi_azimuth
        im_obs['Z1.0'] = gm_im_table_50_filt['Z1.0']
        im_obs['TRI'] = gm_im_table_50_filt['TRI']
        im_obs['TWI'] = gm_im_table_50_filt['TWI']
        im_obs['geoage'] = gm_im_table_50_filt['geoage']
        im_obs['k0'] = gm_im_table_50_filt['k0']
        im_obs['precip'] = gm_im_table_50_filt['precip']
        im_obs[pSAobs_T[iper]] = gm_im_table_50_filt[pSAobs_T[iper]].values

        im_sim = im_obs.copy()
        im_sim[pSAobs_T[iper]] = SA_im[:,iper]

        # compute within-event residuals
        im_dbebar_ab92 = im_sim.copy()
        im_dbe_ab92 = im_sim.copy()
        im_dwbar_ab92 = im_sim.copy()
        im_residual = im_sim.copy()

        for tect_class in ['Crustal','Interface','Slab']:

            # tect class events
            gm_im_table_50_filt_tect_class = gm_im_table_50_filt[gm_im_table_50_filt.tect_class==tect_class]
            idx_tect_class = gm_im_table_50_filt_tect_class.index
            SA_std_tau_tect_class = SA_std_tau[idx_tect_class]
            SA_std_phi_tect_class = SA_std_phi[idx_tect_class]
            im_residual_tect_class = im_residual.iloc[idx_tect_class]
            im_obs_tect_class = im_obs.iloc[idx_tect_class]
            im_sim_tect_class = im_sim.iloc[idx_tect_class]

            period_col = pSAobs_T[iper]   # one period
            res = ab92_fit_one_period(
                    im_obs_tect_class[period_col],           # observed IMs for this period
                    im_sim_tect_class[period_col],           # GMM medians for this period
                    im_residual_tect_class["evid"],          # event IDs
                    SA_std_tau_tect_class[:,iper],
                    SA_std_phi_tect_class[:,iper],
                    center=False                  # set True only if you want to de-bias Δ by its grand mean
            )

            tau_hat, phi_hat = res["tau"], res["phi"]
            im_dbebar_ab92.loc[idx_tect_class, period_col] = res["dbe_bar"].values
            im_dbe_ab92.loc[idx_tect_class, period_col] = res["deltaB"].values
            im_dwbar_ab92.loc[idx_tect_class, period_col]  = res["dw_bar"].values

        if save:
            # save file
            pSA_names = ['PGA','pSA_003','pSA_006','pSA_010','pSA_030','pSA_060','pSA_100','pSA_200','pSA_300','pSA_400','pSA_500','pSA_600','pSA_700','pSA_800','pSA_900','pSA_1000']

            im_dbe_ab92_file = 'NZGMDB_v4p3_dbe_grp1_n3_%s'%pSA_names[iper]
            im_dbebar_ab92_file = 'NZGMDB_v4p3_dbebar_grp1_n3_%s'%pSA_names[iper]
            im_dwbar_ab92_file = 'NZGMDB_v4p3_dwbar_grp1_n3_%s'%pSA_names[iper]

            im_dwbar_ab92.to_csv(outdir / f"{im_dwbar_ab92_file}.csv", index=False)
            im_dbe_ab92.to_csv(outdir / f"{im_dbe_ab92_file}.csv", index=False)
            im_dbebar_ab92.to_csv(outdir / f"{im_dbebar_ab92_file}.csv", index=False)

## 2.4 Computation: Group 2

In [ ]:
if run_imrescalc:
    for iper in tqdm(range(len(gm_im_table_50_grp2)), desc = 'Processing Period: '):
        gm_im_table_50_filt = gm_im_table_50_grp2[iper]

        SA_im = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_tau = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_phi = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))

        for i in tqdm(range(len(gm_im_table_50_filt)), desc = 'Computing IMs'):

                location = Point(
                        0.0, 0.0, 0.0
                )  # Create a dummy location as OQ calculation doesn't use a location
                oq_site = Site(location)
                rup = Properties()
                dists = Properties()

                ##set input parameters
                #set source properties
                rup.mag = np.array([gm_im_table_50_filt.mag[i]])
                rup.rake = np.array([gm_im_table_50_filt.rake[i]])
                rup.dip = np.array([gm_im_table_50_filt.dip[i]])
                rup.ztor = np.array([gm_im_table_50_filt.z_tor[i]])
                rup.width = np.array([gm_im_table_50_filt.f_width[i]])
                rup.hypo_depth = np.array([gm_im_table_50_filt.ev_depth[i]])

                #set site properties (ALWAYS CHECK UNITS THE MODEL USES, e.g. m or km)
                oq_site.vs30 = np.array([gm_im_table_50_filt.Vs30[i]])    #this needs to be in m/s
                oq_site.vs30measured = np.array([False])
                oq_site.z1pt0 = np.array([gm_im_table_50_filt['Z1.0'][i]])    #this needs to be in m
                oq_site.z2pt5 = np.array([gm_im_table_50_filt['Z2.5'][i]])
                oq_site.backarc = np.array([gm_im_table_50_filt['backarc'][i]])
                sites = SiteCollection([oq_site])

                #set distance properties (generally in km)
                dists.rhypo = np.array([gm_im_table_50_filt.r_hyp[i]])
                dists.rrup = np.array([gm_im_table_50_filt.r_rup[i]])
                dists.repi = np.array([gm_im_table_50_filt.r_epi[i]])
                dists.rjb = np.array([gm_im_table_50_filt.r_jb[i]])
                dists.rx = np.array([gm_im_table_50_filt.r_x[i]])
                dists.ry0 = np.array([gm_im_table_50_filt.r_y[i]])

                # choose GMPE
                tect = gm_im_table_50_filt.tect_class.iloc[i]
                if tect == 'Interface':
                    GMPE = K_2020I
                elif tect == 'Slab':
                    GMPE = K_2020S
                else:
                    GMPE = CY_2014

                if iper == 0:
                    imt_obj = imt.PGA()
                else:
                    imt_obj = imt.SA(period=pSA_T[iper])

                m, stdevs = GMPE.get_mean_and_stddevs(SiteCollection([oq_site]), rup, dists, imt_obj, stddev_types)
                SA_im[i,iper]      = np.exp(m[0])
                SA_std[i,iper]     = stdevs[0][0]
                SA_std_tau[i,iper] = stdevs[1][0]
                SA_std_phi[i,iper] = stdevs[2][0]

        # create im_sim/im_obs dataframe
        im_obs = pd.DataFrame()
        im_obs['evid'] = gm_im_table_50_filt.evid
        im_obs['sta'] = gm_im_table_50_filt.sta
        im_obs['sta_lat'] = gm_im_table_50_filt.sta_lat
        im_obs['sta_lon'] = gm_im_table_50_filt.sta_lon
        im_obs['vs30'] = gm_im_table_50_filt.Vs30
        im_obs['epicenter_dist'] = gm_im_table_50_filt.epi_dist
        im_obs['epicenter_azimuth'] = gm_im_table_50_filt.epi_azimuth
        im_obs['Z1.0'] = gm_im_table_50_filt['Z1.0']
        im_obs['TRI'] = gm_im_table_50_filt['TRI']
        im_obs['TWI'] = gm_im_table_50_filt['TWI']
        im_obs['geoage'] = gm_im_table_50_filt['geoage']
        im_obs['k0'] = gm_im_table_50_filt['k0']
        im_obs['precip'] = gm_im_table_50_filt['precip']
        im_obs[pSAobs_T[iper]] = gm_im_table_50_filt[pSAobs_T[iper]].values

        im_sim = im_obs.copy()
        im_sim[pSAobs_T[iper]] = SA_im[:,iper]

        # compute within-event residuals
        im_dbebar_ab92 = im_sim.copy()
        im_dbe_ab92 = im_sim.copy()
        im_dwbar_ab92 = im_sim.copy()
        im_residual = im_sim.copy()

        for tect_class in ['Crustal','Interface','Slab']:

            # tect class events
            gm_im_table_50_filt_tect_class = gm_im_table_50_filt[gm_im_table_50_filt.tect_class==tect_class]
            idx_tect_class = gm_im_table_50_filt_tect_class.index
            SA_std_tau_tect_class = SA_std_tau[idx_tect_class]
            SA_std_phi_tect_class = SA_std_phi[idx_tect_class]
            im_residual_tect_class = im_residual.iloc[idx_tect_class]
            im_obs_tect_class = im_obs.iloc[idx_tect_class]
            im_sim_tect_class = im_sim.iloc[idx_tect_class]

            period_col = pSAobs_T[iper]   # one period
            res = ab92_fit_one_period(
                    im_obs_tect_class[period_col],           # observed IMs for this period
                    im_sim_tect_class[period_col],           # GMM medians for this period
                    im_residual_tect_class["evid"],          # event IDs
                    SA_std_tau_tect_class[:,iper],
                    SA_std_phi_tect_class[:,iper],
                    center=False                  # set True only if you want to de-bias Δ by its grand mean
            )

            tau_hat, phi_hat = res["tau"], res["phi"]
            im_dbebar_ab92.loc[idx_tect_class, period_col] = res["dbe_bar"].values
            im_dbe_ab92.loc[idx_tect_class, period_col] = res["deltaB"].values
            im_dwbar_ab92.loc[idx_tect_class, period_col]  = res["dw_bar"].values

        if save:
            # save file
            pSA_names = ['PGA','pSA_003','pSA_006','pSA_010','pSA_030','pSA_060','pSA_100','pSA_200','pSA_300','pSA_400','pSA_500','pSA_600','pSA_700','pSA_800','pSA_900','pSA_1000']

            im_dbe_ab92_file = 'NZGMDB_v4p3_dbe_grp2_n3_%s'%pSA_names[iper]
            im_dbebar_ab92_file = 'NZGMDB_v4p3_dbebar_grp2_n3_%s'%pSA_names[iper]
            im_dwbar_ab92_file = 'NZGMDB_v4p3_dwbar_grp2_n3_%s'%pSA_names[iper]

            im_dwbar_ab92.to_csv(outdir / f"{im_dwbar_ab92_file}.csv", index=False)
            im_dbe_ab92.to_csv(outdir / f"{im_dbe_ab92_file}.csv", index=False)
            im_dbebar_ab92.to_csv(outdir / f"{im_dbebar_ab92_file}.csv", index=False)

## 2.5 Computation: Group North Island

In [ ]:
if run_imrescalc:
    for iper in tqdm(range(len(gm_im_table_50_ngrp)), desc = 'Processing Period: '):
        gm_im_table_50_filt = gm_im_table_50_ngrp[iper]

        SA_im = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_tau = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_phi = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))

        for i in tqdm(range(len(gm_im_table_50_filt)), desc = 'Computing IMs'):

                location = Point(
                        0.0, 0.0, 0.0
                )  # Create a dummy location as OQ calculation doesn't use a location
                oq_site = Site(location)
                rup = Properties()
                dists = Properties()

                ##set input parameters
                #set source properties
                rup.mag = np.array([gm_im_table_50_filt.mag[i]])
                rup.rake = np.array([gm_im_table_50_filt.rake[i]])
                rup.dip = np.array([gm_im_table_50_filt.dip[i]])
                rup.ztor = np.array([gm_im_table_50_filt.z_tor[i]])
                rup.width = np.array([gm_im_table_50_filt.f_width[i]])
                rup.hypo_depth = np.array([gm_im_table_50_filt.ev_depth[i]])

                #set site properties (ALWAYS CHECK UNITS THE MODEL USES, e.g. m or km)
                oq_site.vs30 = np.array([gm_im_table_50_filt.Vs30[i]])    #this needs to be in m/s
                oq_site.vs30measured = np.array([False])
                oq_site.z1pt0 = np.array([gm_im_table_50_filt['Z1.0'][i]])    #this needs to be in m
                oq_site.z2pt5 = np.array([gm_im_table_50_filt['Z2.5'][i]])
                oq_site.backarc = np.array([gm_im_table_50_filt['backarc'][i]])
                sites = SiteCollection([oq_site])

                #set distance properties (generally in km)
                dists.rhypo = np.array([gm_im_table_50_filt.r_hyp[i]])
                dists.rrup = np.array([gm_im_table_50_filt.r_rup[i]])
                dists.repi = np.array([gm_im_table_50_filt.r_epi[i]])
                dists.rjb = np.array([gm_im_table_50_filt.r_jb[i]])
                dists.rx = np.array([gm_im_table_50_filt.r_x[i]])
                dists.ry0 = np.array([gm_im_table_50_filt.r_y[i]])

                # choose GMPE
                tect = gm_im_table_50_filt.tect_class.iloc[i]
                if tect == 'Interface':
                    GMPE = K_2020I
                elif tect == 'Slab':
                    GMPE = K_2020S
                else:
                    GMPE = CY_2014

                if iper == 0:
                    imt_obj = imt.PGA()
                else:
                    imt_obj = imt.SA(period=pSA_T[iper])

                m, stdevs = GMPE.get_mean_and_stddevs(SiteCollection([oq_site]), rup, dists, imt_obj, stddev_types)
                SA_im[i,iper]      = np.exp(m[0])
                SA_std[i,iper]     = stdevs[0][0]
                SA_std_tau[i,iper] = stdevs[1][0]
                SA_std_phi[i,iper] = stdevs[2][0]

        # create im_sim/im_obs dataframe
        im_obs = pd.DataFrame()
        im_obs['evid'] = gm_im_table_50_filt.evid
        im_obs['sta'] = gm_im_table_50_filt.sta
        im_obs['sta_lat'] = gm_im_table_50_filt.sta_lat
        im_obs['sta_lon'] = gm_im_table_50_filt.sta_lon
        im_obs['vs30'] = gm_im_table_50_filt.Vs30
        im_obs['epicenter_dist'] = gm_im_table_50_filt.epi_dist
        im_obs['epicenter_azimuth'] = gm_im_table_50_filt.epi_azimuth
        im_obs['Z1.0'] = gm_im_table_50_filt['Z1.0']
        im_obs['TRI'] = gm_im_table_50_filt['TRI']
        im_obs['TWI'] = gm_im_table_50_filt['TWI']
        im_obs['geoage'] = gm_im_table_50_filt['geoage']
        im_obs['k0'] = gm_im_table_50_filt['k0']
        im_obs['precip'] = gm_im_table_50_filt['precip']
        im_obs[pSAobs_T[iper]] = gm_im_table_50_filt[pSAobs_T[iper]].values

        im_sim = im_obs.copy()
        im_sim[pSAobs_T[iper]] = SA_im[:,iper]

        # compute within-event residuals
        im_dbebar_ab92 = im_sim.copy()
        im_dbe_ab92 = im_sim.copy()
        im_dwbar_ab92 = im_sim.copy()
        im_residual = im_sim.copy()

        for tect_class in ['Crustal','Interface','Slab']:

            # tect class events
            gm_im_table_50_filt_tect_class = gm_im_table_50_filt[gm_im_table_50_filt.tect_class==tect_class]
            idx_tect_class = gm_im_table_50_filt_tect_class.index
            SA_std_tau_tect_class = SA_std_tau[idx_tect_class]
            SA_std_phi_tect_class = SA_std_phi[idx_tect_class]
            im_residual_tect_class = im_residual.iloc[idx_tect_class]
            im_obs_tect_class = im_obs.iloc[idx_tect_class]
            im_sim_tect_class = im_sim.iloc[idx_tect_class]

            period_col = pSAobs_T[iper]   # one period
            res = ab92_fit_one_period(
                    im_obs_tect_class[period_col],           # observed IMs for this period
                    im_sim_tect_class[period_col],           # GMM medians for this period
                    im_residual_tect_class["evid"],          # event IDs
                    SA_std_tau_tect_class[:,iper],
                    SA_std_phi_tect_class[:,iper],
                    center=False                  # set True only if you want to de-bias Δ by its grand mean
            )

            tau_hat, phi_hat = res["tau"], res["phi"]
            im_dbebar_ab92.loc[idx_tect_class, period_col] = res["dbe_bar"].values
            im_dbe_ab92.loc[idx_tect_class, period_col] = res["deltaB"].values
            im_dwbar_ab92.loc[idx_tect_class, period_col]  = res["dw_bar"].values

        if save:
            # save file
            pSA_names = ['PGA','pSA_003','pSA_006','pSA_010','pSA_030','pSA_060','pSA_100','pSA_200','pSA_300','pSA_400','pSA_500','pSA_600','pSA_700','pSA_800','pSA_900','pSA_1000']

            im_dbe_ab92_file = 'NZGMDB_v4p3_dbe_grpn_n3_%s'%pSA_names[iper]
            im_dbebar_ab92_file = 'NZGMDB_v4p3_dbebar_grpn_n3_%s'%pSA_names[iper]
            im_dwbar_ab92_file = 'NZGMDB_v4p3_dwbar_grpn_n3_%s'%pSA_names[iper]

            im_dwbar_ab92.to_csv(outdir / f"{im_dwbar_ab92_file}.csv", index=False)
            im_dbe_ab92.to_csv(outdir / f"{im_dbe_ab92_file}.csv", index=False)
            im_dbebar_ab92.to_csv(outdir / f"{im_dbebar_ab92_file}.csv", index=False)

## 2.6 Computation: Group South Island

In [ ]:
if run_imrescalc:
    for iper in tqdm(range(len(gm_im_table_50_sgrp)), desc = 'Processing Period: '):
        gm_im_table_50_filt = gm_im_table_50_sgrp[iper]

        SA_im = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_tau = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
        SA_std_phi = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))

        for i in tqdm(range(len(gm_im_table_50_filt)), desc = 'Computing IMs'):

                location = Point(
                        0.0, 0.0, 0.0
                )  # Create a dummy location as OQ calculation doesn't use a location
                oq_site = Site(location)
                rup = Properties()
                dists = Properties()

                ##set input parameters
                #set source properties
                rup.mag = np.array([gm_im_table_50_filt.mag[i]])
                rup.rake = np.array([gm_im_table_50_filt.rake[i]])
                rup.dip = np.array([gm_im_table_50_filt.dip[i]])
                rup.ztor = np.array([gm_im_table_50_filt.z_tor[i]])
                rup.width = np.array([gm_im_table_50_filt.f_width[i]])
                rup.hypo_depth = np.array([gm_im_table_50_filt.ev_depth[i]])

                #set site properties (ALWAYS CHECK UNITS THE MODEL USES, e.g. m or km)
                oq_site.vs30 = np.array([gm_im_table_50_filt.Vs30[i]])    #this needs to be in m/s
                oq_site.vs30measured = np.array([False])
                oq_site.z1pt0 = np.array([gm_im_table_50_filt['Z1.0'][i]])    #this needs to be in m
                oq_site.z2pt5 = np.array([gm_im_table_50_filt['Z2.5'][i]])
                oq_site.backarc = np.array([gm_im_table_50_filt['backarc'][i]])
                sites = SiteCollection([oq_site])

                #set distance properties (generally in km)
                dists.rhypo = np.array([gm_im_table_50_filt.r_hyp[i]])
                dists.rrup = np.array([gm_im_table_50_filt.r_rup[i]])
                dists.repi = np.array([gm_im_table_50_filt.r_epi[i]])
                dists.rjb = np.array([gm_im_table_50_filt.r_jb[i]])
                dists.rx = np.array([gm_im_table_50_filt.r_x[i]])
                dists.ry0 = np.array([gm_im_table_50_filt.r_y[i]])

                # choose GMPE
                tect = gm_im_table_50_filt.tect_class.iloc[i]
                if tect == 'Interface':
                    GMPE = K_2020I
                elif tect == 'Slab':
                    GMPE = K_2020S
                else:
                    GMPE = CY_2014

                if iper == 0:
                    imt_obj = imt.PGA()
                else:
                    imt_obj = imt.SA(period=pSA_T[iper])

                m, stdevs = GMPE.get_mean_and_stddevs(SiteCollection([oq_site]), rup, dists, imt_obj, stddev_types)
                SA_im[i,iper]      = np.exp(m[0])
                SA_std[i,iper]     = stdevs[0][0]
                SA_std_tau[i,iper] = stdevs[1][0]
                SA_std_phi[i,iper] = stdevs[2][0]

        # create im_sim/im_obs dataframe
        im_obs = pd.DataFrame()
        im_obs['evid'] = gm_im_table_50_filt.evid
        im_obs['sta'] = gm_im_table_50_filt.sta
        im_obs['sta_lat'] = gm_im_table_50_filt.sta_lat
        im_obs['sta_lon'] = gm_im_table_50_filt.sta_lon
        im_obs['vs30'] = gm_im_table_50_filt.Vs30
        im_obs['epicenter_dist'] = gm_im_table_50_filt.epi_dist
        im_obs['epicenter_azimuth'] = gm_im_table_50_filt.epi_azimuth
        im_obs['Z1.0'] = gm_im_table_50_filt['Z1.0']
        im_obs['TRI'] = gm_im_table_50_filt['TRI']
        im_obs['TWI'] = gm_im_table_50_filt['TWI']
        im_obs['geoage'] = gm_im_table_50_filt['geoage']
        im_obs['k0'] = gm_im_table_50_filt['k0']
        im_obs['precip'] = gm_im_table_50_filt['precip']
        im_obs[pSAobs_T[iper]] = gm_im_table_50_filt[pSAobs_T[iper]].values

        im_sim = im_obs.copy()
        im_sim[pSAobs_T[iper]] = SA_im[:,iper]

        # compute within-event residuals
        im_dbebar_ab92 = im_sim.copy()
        im_dbe_ab92 = im_sim.copy()
        im_dwbar_ab92 = im_sim.copy()
        im_residual = im_sim.copy()

        for tect_class in ['Crustal','Interface','Slab']:

            # tect class events
            gm_im_table_50_filt_tect_class = gm_im_table_50_filt[gm_im_table_50_filt.tect_class==tect_class]
            idx_tect_class = gm_im_table_50_filt_tect_class.index
            SA_std_tau_tect_class = SA_std_tau[idx_tect_class]
            SA_std_phi_tect_class = SA_std_phi[idx_tect_class]
            im_residual_tect_class = im_residual.iloc[idx_tect_class]
            im_obs_tect_class = im_obs.iloc[idx_tect_class]
            im_sim_tect_class = im_sim.iloc[idx_tect_class]

            period_col = pSAobs_T[iper]   # one period
            res = ab92_fit_one_period(
                    im_obs_tect_class[period_col],           # observed IMs for this period
                    im_sim_tect_class[period_col],           # GMM medians for this period
                    im_residual_tect_class["evid"],          # event IDs
                    SA_std_tau_tect_class[:,iper],
                    SA_std_phi_tect_class[:,iper],
                    center=False                  # set True only if you want to de-bias Δ by its grand mean
            )

            tau_hat, phi_hat = res["tau"], res["phi"]
            im_dbebar_ab92.loc[idx_tect_class, period_col] = res["dbe_bar"].values
            im_dbe_ab92.loc[idx_tect_class, period_col] = res["deltaB"].values
            im_dwbar_ab92.loc[idx_tect_class, period_col]  = res["dw_bar"].values

        if save:
            # save file
            pSA_names = ['PGA','pSA_003','pSA_006','pSA_010','pSA_030','pSA_060','pSA_100','pSA_200','pSA_300','pSA_400','pSA_500','pSA_600','pSA_700','pSA_800','pSA_900','pSA_1000']

            im_dbe_ab92_file = 'NZGMDB_v4p3_dbe_grps_n3_%s'%pSA_names[iper]
            im_dbebar_ab92_file = 'NZGMDB_v4p3_dbebar_grps_n3_%s'%pSA_names[iper]
            im_dwbar_ab92_file = 'NZGMDB_v4p3_dwbar_grps_n3_%s'%pSA_names[iper]

            im_dwbar_ab92.to_csv(outdir / f"{im_dwbar_ab92_file}.csv", index=False)
            im_dbe_ab92.to_csv(outdir / f"{im_dbe_ab92_file}.csv", index=False)
            im_dbebar_ab92.to_csv(outdir / f"{im_dbebar_ab92_file}.csv", index=False)

## 2.7 Computation: All data at max period

In [ ]:
if run_imrescalc:

    gm_im_table_50_filt = gm_im_table_50_sa[11] # desired max period

    SA_im = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
    SA_std = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
    SA_std_tau = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))
    SA_std_phi = np.zeros((len(gm_im_table_50_filt), len(pSA_T)))

    for iper in tqdm(range(len(pSAobs_T)), desc = 'Processing Period: '):
        for i in tqdm(range(len(gm_im_table_50_filt)), desc = 'Computing IMs'):

                location = Point(
                        0.0, 0.0, 0.0
                )  # Create a dummy location as OQ calculation doesn't use a location
                oq_site = Site(location)
                rup = Properties()
                dists = Properties()

                ##set input parameters
                #set source properties
                rup.mag = np.array([gm_im_table_50_filt.mag[i]])
                rup.rake = np.array([gm_im_table_50_filt.rake[i]])
                rup.dip = np.array([gm_im_table_50_filt.dip[i]])
                rup.ztor = np.array([gm_im_table_50_filt.z_tor[i]])
                rup.width = np.array([gm_im_table_50_filt.f_width[i]])
                rup.hypo_depth = np.array([gm_im_table_50_filt.ev_depth[i]])

                #set site properties (ALWAYS CHECK UNITS THE MODEL USES, e.g. m or km)
                oq_site.vs30 = np.array([gm_im_table_50_filt.Vs30[i]])    #this needs to be in m/s
                oq_site.vs30measured = np.array([False])
                oq_site.z1pt0 = np.array([gm_im_table_50_filt['Z1.0'][i]])    #this needs to be in m
                oq_site.z2pt5 = np.array([gm_im_table_50_filt['Z2.5'][i]])
                oq_site.backarc = np.array([gm_im_table_50_filt['backarc'][i]])
                sites = SiteCollection([oq_site])

                #set distance properties (generally in km)
                dists.rhypo = np.array([gm_im_table_50_filt.r_hyp[i]])
                dists.rrup = np.array([gm_im_table_50_filt.r_rup[i]])
                dists.repi = np.array([gm_im_table_50_filt.r_epi[i]])
                dists.rjb = np.array([gm_im_table_50_filt.r_jb[i]])
                dists.rx = np.array([gm_im_table_50_filt.r_x[i]])
                dists.ry0 = np.array([gm_im_table_50_filt.r_y[i]])

                if gm_im_table_50_filt.tect_class[i] == 'Interface':
                    GMPE = K_2020I

                    if iper == 0:
                        im_PGA = imt.PGA()
                        m_PGA, stdev_PGA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_PGA, stddev_types)
                        SA_im[i, 0] = np.exp(m_PGA[0])
                        SA_std[i, 0] = stdev_PGA[0][0]
                        SA_std_tau[i, 0] = stdev_PGA[1][0]
                        SA_std_phi[i, 0] = stdev_PGA[2][0]
                    else:
                        im_SA = imt.SA(period=pSA_T[iper])
                        m_SA, stdev_SA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_SA, stddev_types)
                        SA_im[i, iper] = np.exp(m_SA[0])
                        SA_std[i, iper] = stdev_SA[0][0]
                        SA_std_tau[i, iper] = stdev_SA[1][0]
                        SA_std_phi[i, iper] = stdev_SA[2][0]

                elif gm_im_table_50_filt.tect_class[i] == 'Slab':
                    GMPE = K_2020S

                    if iper == 0:
                        im_PGA = imt.PGA()
                        m_PGA, stdev_PGA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_PGA, stddev_types)
                        SA_im[i, 0] = np.exp(m_PGA[0])
                        SA_std[i, 0] = stdev_PGA[0][0]
                        SA_std_tau[i, 0] = stdev_PGA[1][0]
                        SA_std_phi[i, 0] = stdev_PGA[2][0]
                    else:
                        im_SA = imt.SA(period=pSA_T[iper])
                        m_SA, stdev_SA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_SA, stddev_types)
                        SA_im[i, iper] = np.exp(m_SA[0])
                        SA_std[i, iper] = stdev_SA[0][0]
                        SA_std_tau[i, iper] = stdev_SA[1][0]
                        SA_std_phi[i, iper] = stdev_SA[2][0]

                else:
                    GMPE = CY_2014

                    if iper == 0:
                        im_PGA = imt.PGA()
                        m_PGA, stdev_PGA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_PGA, stddev_types)
                        SA_im[i, 0] = np.exp(m_PGA[0])
                        SA_std[i, 0] = stdev_PGA[0][0]
                        SA_std_tau[i, 0] = stdev_PGA[1][0]
                        SA_std_phi[i, 0] = stdev_PGA[2][0]
                    else:
                        im_SA = imt.SA(period=pSA_T[iper])
                        m_SA, stdev_SA = GMPE.get_mean_and_stddevs(sites, rup, dists, im_SA, stddev_types)
                        SA_im[i, iper] = np.exp(m_SA[0])
                        SA_std[i, iper] = stdev_SA[0][0]
                        SA_std_tau[i, iper] = stdev_SA[1][0]
                        SA_std_phi[i, iper] = stdev_SA[2][0]

    # create im_sim/im_obs dataframe
    im_obs = pd.DataFrame()
    im_obs['evid'] = gm_im_table_50_filt.evid
    im_obs['sta'] = gm_im_table_50_filt.sta
    im_obs['sta_lat'] = gm_im_table_50_filt.sta_lat
    im_obs['sta_lon'] = gm_im_table_50_filt.sta_lon
    im_obs['vs30'] = gm_im_table_50_filt.Vs30
    im_obs['epicenter_dist'] = gm_im_table_50_filt.epi_dist
    im_obs['epicenter_azimuth'] = gm_im_table_50_filt.epi_azimuth
    im_obs['Z1.0'] = gm_im_table_50_filt['Z1.0']
    im_obs['TRI'] = gm_im_table_50_filt['TRI']
    im_obs['TWI'] = gm_im_table_50_filt['TWI']
    im_obs['geoage'] = gm_im_table_50_filt['geoage']
    im_obs['k0'] = gm_im_table_50_filt['k0']
    im_obs['precip'] = gm_im_table_50_filt['precip']
    im_obs[pSAobs_T] = gm_im_table_50_filt[pSAobs_T].values

    im_sim = im_obs.copy()
    im_sim[pSAobs_T] = SA_im

    # compute within-event residuals
    im_dbebar_ab92 = im_sim.copy()
    im_dbe_ab92 = im_sim.copy()
    im_dwbar_ab92 = im_sim.copy()
    im_residual = im_sim.copy()

    for tect_class in ['Crustal','Interface','Slab']:

        # tect class events
        gm_im_table_50_filt_tect_class = gm_im_table_50_filt[gm_im_table_50_filt.tect_class==tect_class]
        idx_tect_class = gm_im_table_50_filt_tect_class.index
        SA_std_tau_tect_class = SA_std_tau[idx_tect_class]
        SA_std_phi_tect_class = SA_std_phi[idx_tect_class]
        im_residual_tect_class = im_residual.iloc[idx_tect_class]
        im_obs_tect_class = im_obs.iloc[idx_tect_class]
        im_sim_tect_class = im_sim.iloc[idx_tect_class]

        for i in tqdm(range(len(pSAobs_T)), desc = '%s: '%tect_class):
            period_col = pSAobs_T[i]  # one period
            res = ab92_fit_one_period(
                im_obs_tect_class[period_col],           # observed IMs for this period
                im_sim_tect_class[period_col],           # GMM medians for this period
                im_residual_tect_class["evid"],          # event IDs
                SA_std_tau_tect_class[:,i],
                SA_std_phi_tect_class[:,i],
                center=False                  # set True only if you want to de-bias Δ by its grand mean
            )

            tau_hat, phi_hat = res["tau"], res["phi"]
            im_dbebar_ab92.loc[idx_tect_class, period_col] = res["dbe_bar"].values
            im_dbe_ab92.loc[idx_tect_class, period_col] = res["deltaB"].values
            im_dwbar_ab92.loc[idx_tect_class, period_col]  = res["dw_bar"].values

    if save:

        im_dbe_ab92_file = 'NZGMDB_v4p3_dbe_T6_n3'
        im_dbebar_ab92_file = 'NZGMDB_v4p3_dbebar_T6_n3'
        im_dwbar_ab92_file = 'NZGMDB_v4p3_dwbar_T6_n3'

        im_dwbar_ab92.to_csv(outdir / f"{im_dwbar_ab92_file}.csv", index=False)
        im_dbe_ab92.to_csv(outdir / f"{im_dbe_ab92_file}.csv", index=False)
        im_dbebar_ab92.to_csv(outdir / f"{im_dbebar_ab92_file}.csv", index=False)

# 3.0 Plot station pairs

## 3.1 Function definitions

In [ ]:
def getEucDistance(X):
    '''
    Computes a matrix with the Euclidean distance from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with distance from each row in X to all
        the other rows (nk x nk)
    '''
    sq_dist = (np.square(np.reshape(X[:,0],[-1,1])) +
        np.square(np.reshape(X[:,0],[1,-1])) -
        2 * (np.reshape(X[:,0],[-1,1]) * np.reshape(X[:,0],[1,-1])) *
        np.cos(np.abs(np.reshape(X[:,1],[1,-1]) - np.reshape(X[:,1],[-1,1]) )) )
    sq_dist = np.clip(sq_dist, 0, np.inf)
    dist = np.sqrt(sq_dist)
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def getAngDistance(X):
    '''
    Computes a matrix with the angular distance from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with Angular distance from each
        row in X to all the other rows (nk x nk)
    '''
    cos_angle = np.cos( np.abs(np.reshape(X[:,1],[1,-1]) -
                                np.reshape(X[:,1],[-1,1]) ))
    dist =  np.arccos(np.clip(cos_angle, -1, 1))
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def getSoilDissimilarity(X):
    '''
    Computes a matrix with the soil dissimilarity from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with soil dissimilarities from each
        row in X to all the other rows (nk x nk)
    '''
    sq_dist = np.square(np.reshape(X[:,2], [-1,1]) - np.reshape(X[:,2], [1,-1]))
    sq_dist = np.clip(sq_dist, 0, np.inf)
    dist = np.sqrt(sq_dist)
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def getBasinDissimilarity(X):
    '''
    Computes a matrix with the basin dissimilarity from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with soil dissimilarities from each
        row in X to all the other rows (nk x nk)
    '''
    sq_dist = np.square(np.reshape(X[:,3], [-1,1]) - np.reshape(X[:,3], [1,-1]))
    sq_dist = np.clip(sq_dist, 0, np.inf)
    dist = np.sqrt(sq_dist)
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def getTopoDissimilarity(X):
    '''
    Computes a matrix with the basin dissimilarity from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with soil dissimilarities from each
        row in X to all the other rows (nk x nk)
    '''
    sq_dist = np.square(np.reshape(X[:,4], [-1,1]) - np.reshape(X[:,4], [1,-1]))
    sq_dist = np.clip(sq_dist, 0, np.inf)
    dist = np.sqrt(sq_dist)
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def get_npairs_euclS(X, bin_s, bin_e):
    dist_e = getEucDistance(X)
    dist_s = getSoilDissimilarity(X)
    nPairs = np.zeros((len(bin_e), len(bin_s)))
    for i, be in enumerate(bin_e):
        for j, bs in enumerate(bin_s):
            nP = np.sum( (dist_e > be[0]) & (dist_e <= be[1])
                & (dist_s > bs[0]) & (dist_s <= bs[1])) /2
            nPairs[i, j] = int(nP)
    return nPairs

def get_npairs_euclZ(X, bin_s, bin_e):
    dist_e = getEucDistance(X)
    dist_s = getBasinDissimilarity(X)
    nPairs = np.zeros((len(bin_e), len(bin_s)))
    for i, be in enumerate(bin_e):
        for j, bs in enumerate(bin_s):
            nP = np.sum( (dist_e > be[0]) & (dist_e <= be[1])
                & (dist_s > bs[0]) & (dist_s <= bs[1])) /2
            nPairs[i, j] = int(nP)
    return nPairs

def get_npairs_euclT(X, bin_s, bin_e):
    dist_e = getEucDistance(X)
    dist_s = getTopoDissimilarity(X)
    nPairs = np.zeros((len(bin_e), len(bin_s)))
    for i, be in enumerate(bin_e):
        for j, bs in enumerate(bin_s):
            nP = np.sum( (dist_e > be[0]) & (dist_e <= be[1])
                & (dist_s > bs[0]) & (dist_s <= bs[1])) /2
            nPairs[i, j] = int(nP)
    return nPairs

def get_npairs_angeucl(X, bin_a, bin_e):
    dist_e = getEucDistance(X)
    dist_a = getAngDistance(X)
    nPairs = np.zeros((len(bin_e), len(bin_a)))
    for i, be in enumerate(bin_e):
        for j, ba in enumerate(bin_a):
            ba0 = ba[0]/180; ba1 = ba[1]/180
            nP = np.sum( (dist_e > be[0]) & (dist_e <= be[1])
                & (dist_a > ba0) & (dist_a <= ba1)) /2
            nPairs[i, j] = int(nP)
    return nPairs

def outline_threshold_cells(ax, x_edges, y_edges, counts_xy, threshold=30,
                            color="black", linewidth=1.0, zorder=10):
    """
    Draw boundary around cells where counts > threshold.

    Assumes counts_xy has shape:
        (len(x_edges)-1, len(y_edges)-1)

    In your case:
        x = Euclidean distance bins
        y = angular/site/topographic/etc. bins

    Since your nP arrays are stored as [Euclidean bin, feature bin],
    this function transposes internally for plotting.
    """
    ok = np.asarray(counts_xy).T < threshold  # now shape is [ybin, xbin]

    ny, nx = ok.shape
    segments = []

    for iy in range(ny):
        for ix in range(nx):
            if not ok[iy, ix]:
                continue

            x0, x1 = x_edges[ix], x_edges[ix + 1]
            y0, y1 = y_edges[iy], y_edges[iy + 1]

            # left boundary
            if ix == 0 or not ok[iy, ix - 1]:
                segments.append([(x0, y0), (x0, y1)])

            # right boundary
            if ix == nx - 1 or not ok[iy, ix + 1]:
                segments.append([(x1, y0), (x1, y1)])

            # bottom boundary
            if iy == 0 or not ok[iy - 1, ix]:
                segments.append([(x0, y0), (x1, y0)])

            # top boundary
            if iy == ny - 1 or not ok[iy + 1, ix]:
                segments.append([(x0, y1), (x1, y1)])

    if segments:
        lc = LineCollection(
            segments,
            colors=color,
            linewidths=linewidth,
            zorder=zorder
        )
        ax.add_collection(lc)

## 3.2 Computation pairs per period: Full database

In [ ]:
# Specify bins
bin_t = [(i, i+10) for i in np.arange(0,100,10)]
bin_z = [(i, i+50) for i in np.arange(0,500,50)]
bin_s = [(i, i+50) for i in np.arange(0,500,50)]
bin_a = [(i, i+20) for i in np.arange(0,180,20)]
bin_e = [(i, i+5) for i in np.arange(0,40,5)]

# define edges
e_edges = np.array([b[0] for b in bin_e] + [bin_e[-1][1]])
a_edges = np.array([b[0] for b in bin_a] + [bin_a[-1][1]])
s_edges = np.array([b[0] for b in bin_s] + [bin_s[-1][1]])
z_edges = np.array([b[0] for b in bin_z] + [bin_z[-1][1]])
t_edges = np.array([b[0] for b in bin_t] + [bin_t[-1][1]])

# outlien threshold
threshold = 30

for ip in [6,11]:
    # Loop over all events
    nP_ET = np.zeros((len(bin_e), len(bin_t)))
    nP_EZ = np.zeros((len(bin_e), len(bin_z)))
    nP_ES = np.zeros((len(bin_e), len(bin_s)))
    nP_EA = np.zeros((len(bin_e), len(bin_a)))
    for _, group in gm_im_table_50_sa[ip].groupby('evid'):
        Xtest = group[['epi_dist', 'epi_azimuth', 'Vs30','Z1.0','TRI']].values
        nP_ET += get_npairs_euclT(Xtest, bin_t, bin_e).astype(int)
        nP_EZ += get_npairs_euclZ(Xtest, bin_z, bin_e).astype(int)
        nP_ES += get_npairs_euclS(Xtest, bin_s, bin_e).astype(int)
        nP_EA += get_npairs_angeucl(Xtest, bin_a, bin_e).astype(int)

    # plot
    fig, axs = plt.subplots(1,4, figsize = (12,3), dpi = 200, constrained_layout = True)
    # Compute bin means for plotting
    mbt = np.stack([np.mean(k) for k in bin_t] )
    mbz = np.stack([np.mean(k) for k in bin_z] )
    mbs = np.stack([np.mean(k) for k in bin_s] )
    mbe = np.stack([np.mean(k) for k in bin_e] )
    mba = np.stack([np.mean(k) for k in bin_a] )
    cmap = plt.get_cmap('viridis')
    colors = cmap(np.linspace(0,1,8))
    cmap = mpl.colors.ListedColormap(colors)
    # EA
    ax = axs[0]
    Mba, Mbe = np.meshgrid(mba,mbe)
    im = ax.pcolormesh(Mbe, Mba, nP_EA, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))
    im.set_linewidth(0.001)

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=a_edges,
        counts_xy=nP_EA,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,200,20))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Angular distance $d_{\mathrm{A}}$ [°]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ES
    ax = axs[1]
    Mbs, Mbe = np.meshgrid(mbs,mbe)
    im = ax.pcolormesh(Mbe, Mbs, nP_ES, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
    ax,
    x_edges=e_edges,
    y_edges=s_edges,
    counts_xy=nP_ES,
    threshold=threshold,
    color="red",
    linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Soil dissimilarity $d_{\mathrm{S}}$ [m/s]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # EZ
    ax = axs[2]
    Mbz, Mbe = np.meshgrid(mbz,mbe)
    im = ax.pcolormesh(Mbe, Mbz, nP_EZ, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=z_edges,
        counts_xy=nP_EZ,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Basin dissimilarity $d_{\mathrm{Z1.0}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ET
    ax = axs[3]
    Mbt, Mbe = np.meshgrid(mbt,mbe)
    im = ax.pcolormesh(Mbe, Mbt, nP_ET, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=t_edges,
        counts_xy=nP_ET,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,110,10))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Topographic dissimilarity $d_{\mathrm{TRI}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')

    #colorbar
    cbar = fig.colorbar(im, ax=ax, orientation="vertical",
            ticks=[3, 10, 30, 100, 300, 1000, 3000],
            label='Number of station pairs', drawedges=True)
    cbar.outline.set_linewidth(0.4)
    cbar.outline.set_color('black')
    cbar.dividers.set_linewidth(0.4)
    cbar.dividers.set_color('black')
    cbar.ax.minorticks_off()
    cbar.ax.set_yticklabels(['3', '10','30','100','300','1000', '3000'])
    for a in axs:
        a.grid(True, linewidth=0.3, color='black')
    plt.show()

## 3.3 Computation pairs per period: Group 1

In [ ]:
# Specify bins
bin_t = [(i, i+10) for i in np.arange(0,100,10)]
bin_z = [(i, i+50) for i in np.arange(0,500,50)]
bin_s = [(i, i+50) for i in np.arange(0,500,50)]
bin_a = [(i, i+20) for i in np.arange(0,180,20)]
bin_e = [(i, i+5) for i in np.arange(0,40,5)]

# define edges
e_edges = np.array([b[0] for b in bin_e] + [bin_e[-1][1]])
a_edges = np.array([b[0] for b in bin_a] + [bin_a[-1][1]])
s_edges = np.array([b[0] for b in bin_s] + [bin_s[-1][1]])
z_edges = np.array([b[0] for b in bin_z] + [bin_z[-1][1]])
t_edges = np.array([b[0] for b in bin_t] + [bin_t[-1][1]])

# outlien threshold
threshold = 30

for ip in [6,11]:
    # Loop over all events
    nP_ET = np.zeros((len(bin_e), len(bin_t)))
    nP_EZ = np.zeros((len(bin_e), len(bin_z)))
    nP_ES = np.zeros((len(bin_e), len(bin_s)))
    nP_EA = np.zeros((len(bin_e), len(bin_a)))
    for _, group in gm_im_table_50_grp1[ip].groupby('evid'):
        Xtest = group[['epi_dist', 'epi_azimuth', 'Vs30','Z1.0','TRI']].values
        nP_ET += get_npairs_euclT(Xtest, bin_t, bin_e).astype(int)
        nP_EZ += get_npairs_euclZ(Xtest, bin_z, bin_e).astype(int)
        nP_ES += get_npairs_euclS(Xtest, bin_s, bin_e).astype(int)
        nP_EA += get_npairs_angeucl(Xtest, bin_a, bin_e).astype(int)

    # plot
    fig, axs = plt.subplots(1,4, figsize = (12,3), dpi = 200, constrained_layout = True)
    # Compute bin means for plotting
    mbt = np.stack([np.mean(k) for k in bin_t] )
    mbz = np.stack([np.mean(k) for k in bin_z] )
    mbs = np.stack([np.mean(k) for k in bin_s] )
    mbe = np.stack([np.mean(k) for k in bin_e] )
    mba = np.stack([np.mean(k) for k in bin_a] )
    cmap = plt.get_cmap('viridis')
    colors = cmap(np.linspace(0,1,8))
    cmap = mpl.colors.ListedColormap(colors)
    # EA
    ax = axs[0]
    Mba, Mbe = np.meshgrid(mba,mbe)
    im = ax.pcolormesh(Mbe, Mba, nP_EA, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))
    im.set_linewidth(0.001)

    threshold = 30

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=a_edges,
        counts_xy=nP_EA,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,200,20))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Angular distance $d_{\mathrm{A}}$ [°]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ES
    ax = axs[1]
    Mbs, Mbe = np.meshgrid(mbs,mbe)
    im = ax.pcolormesh(Mbe, Mbs, nP_ES, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=s_edges,
        counts_xy=nP_ES,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Soil dissimilarity $d_{\mathrm{S}}$ [m/s]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # EZ
    ax = axs[2]
    Mbz, Mbe = np.meshgrid(mbz,mbe)
    im = ax.pcolormesh(Mbe, Mbz, nP_EZ, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=z_edges,
        counts_xy=nP_EZ,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Basin dissimilarity $d_{\mathrm{Z1.0}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ET
    ax = axs[3]
    Mbt, Mbe = np.meshgrid(mbt,mbe)
    im = ax.pcolormesh(Mbe, Mbt, nP_ET, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=t_edges,
        counts_xy=nP_ET,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,110,10))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Topographic dissimilarity $d_{\mathrm{TRI}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')

    #colorbar
    cbar = fig.colorbar(im, ax=ax, orientation="vertical",
            ticks=[3, 10, 30, 100, 300, 1000, 3000],
            label='Number of station pairs', drawedges=True)
    cbar.outline.set_linewidth(0.4)
    cbar.outline.set_color('black')
    cbar.dividers.set_linewidth(0.4)
    cbar.dividers.set_color('black')
    cbar.ax.minorticks_off()
    cbar.ax.set_yticklabels(['3', '10','30','100','300','1000', '3000'])
    for a in axs:
        a.grid(True, linewidth=0.3, color='black')
    plt.show()

## 3.4 Computation pairs per period: Group 2

In [ ]:
# Specify bins
bin_t = [(i, i+10) for i in np.arange(0,100,10)]
bin_z = [(i, i+50) for i in np.arange(0,500,50)]
bin_s = [(i, i+50) for i in np.arange(0,500,50)]
bin_a = [(i, i+20) for i in np.arange(0,180,20)]
bin_e = [(i, i+5) for i in np.arange(0,40,5)]

# define edges
e_edges = np.array([b[0] for b in bin_e] + [bin_e[-1][1]])
a_edges = np.array([b[0] for b in bin_a] + [bin_a[-1][1]])
s_edges = np.array([b[0] for b in bin_s] + [bin_s[-1][1]])
z_edges = np.array([b[0] for b in bin_z] + [bin_z[-1][1]])
t_edges = np.array([b[0] for b in bin_t] + [bin_t[-1][1]])

# outlien threshold
threshold = 30

for ip in [6,11]:
    # Loop over all events
    nP_ET = np.zeros((len(bin_e), len(bin_t)))
    nP_EZ = np.zeros((len(bin_e), len(bin_z)))
    nP_ES = np.zeros((len(bin_e), len(bin_s)))
    nP_EA = np.zeros((len(bin_e), len(bin_a)))
    for _, group in gm_im_table_50_grp2[ip].groupby('evid'):
        Xtest = group[['epi_dist', 'epi_azimuth', 'Vs30','Z1.0','TRI']].values
        nP_ET += get_npairs_euclT(Xtest, bin_t, bin_e).astype(int)
        nP_EZ += get_npairs_euclZ(Xtest, bin_z, bin_e).astype(int)
        nP_ES += get_npairs_euclS(Xtest, bin_s, bin_e).astype(int)
        nP_EA += get_npairs_angeucl(Xtest, bin_a, bin_e).astype(int)

    # plot
    fig, axs = plt.subplots(1,4, figsize = (12,3), dpi = 200, constrained_layout = True)
    # Compute bin means for plotting
    mbt = np.stack([np.mean(k) for k in bin_t] )
    mbz = np.stack([np.mean(k) for k in bin_z] )
    mbs = np.stack([np.mean(k) for k in bin_s] )
    mbe = np.stack([np.mean(k) for k in bin_e] )
    mba = np.stack([np.mean(k) for k in bin_a] )
    cmap = plt.get_cmap('viridis')
    colors = cmap(np.linspace(0,1,8))
    cmap = mpl.colors.ListedColormap(colors)
    # EA
    ax = axs[0]
    Mba, Mbe = np.meshgrid(mba,mbe)
    im = ax.pcolormesh(Mbe, Mba, nP_EA, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))
    im.set_linewidth(0.001)

    threshold = 30

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=a_edges,
        counts_xy=nP_EA,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,200,20))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Angular distance $d_{\mathrm{A}}$ [°]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ES
    ax = axs[1]
    Mbs, Mbe = np.meshgrid(mbs,mbe)
    im = ax.pcolormesh(Mbe, Mbs, nP_ES, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=s_edges,
        counts_xy=nP_ES,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Soil dissimilarity $d_{\mathrm{S}}$ [m/s]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # EZ
    ax = axs[2]
    Mbz, Mbe = np.meshgrid(mbz,mbe)
    im = ax.pcolormesh(Mbe, Mbz, nP_EZ, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=z_edges,
        counts_xy=nP_EZ,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Basin dissimilarity $d_{\mathrm{Z1.0}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ET
    ax = axs[3]
    Mbt, Mbe = np.meshgrid(mbt,mbe)
    im = ax.pcolormesh(Mbe, Mbt, nP_ET, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=t_edges,
        counts_xy=nP_ET,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,110,10))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Topographic dissimilarity $d_{\mathrm{TRI}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')

    #colorbar
    cbar = fig.colorbar(im, ax=ax, orientation="vertical",
            ticks=[3, 10, 30, 100, 300, 1000, 3000],
            label='Number of station pairs', drawedges=True)
    cbar.outline.set_linewidth(0.4)
    cbar.outline.set_color('black')
    cbar.dividers.set_linewidth(0.4)
    cbar.dividers.set_color('black')
    cbar.ax.minorticks_off()
    cbar.ax.set_yticklabels(['3', '10','30','100','300','1000', '3000'])
    for a in axs:
        a.grid(True, linewidth=0.3, color='black')
    plt.show()

## 3.5 Computation pairs per period: North Island Group

In [ ]:
# Specify bins
bin_t = [(i, i+10) for i in np.arange(0,100,10)]
bin_z = [(i, i+50) for i in np.arange(0,500,50)]
bin_s = [(i, i+50) for i in np.arange(0,500,50)]
bin_a = [(i, i+20) for i in np.arange(0,180,20)]
bin_e = [(i, i+5) for i in np.arange(0,40,5)]

# define edges
e_edges = np.array([b[0] for b in bin_e] + [bin_e[-1][1]])
a_edges = np.array([b[0] for b in bin_a] + [bin_a[-1][1]])
s_edges = np.array([b[0] for b in bin_s] + [bin_s[-1][1]])
z_edges = np.array([b[0] for b in bin_z] + [bin_z[-1][1]])
t_edges = np.array([b[0] for b in bin_t] + [bin_t[-1][1]])

# outlien threshold
threshold = 30

for ip in [6,11]:
    # Loop over all events
    nP_ET = np.zeros((len(bin_e), len(bin_t)))
    nP_EZ = np.zeros((len(bin_e), len(bin_z)))
    nP_ES = np.zeros((len(bin_e), len(bin_s)))
    nP_EA = np.zeros((len(bin_e), len(bin_a)))
    for _, group in gm_im_table_50_ngrp[ip].groupby('evid'):
        Xtest = group[['epi_dist', 'epi_azimuth', 'Vs30','Z1.0','TRI']].values
        nP_ET += get_npairs_euclT(Xtest, bin_t, bin_e).astype(int)
        nP_EZ += get_npairs_euclZ(Xtest, bin_z, bin_e).astype(int)
        nP_ES += get_npairs_euclS(Xtest, bin_s, bin_e).astype(int)
        nP_EA += get_npairs_angeucl(Xtest, bin_a, bin_e).astype(int)

    # plot
    fig, axs = plt.subplots(1,4, figsize = (12,3), dpi = 200, constrained_layout = True)
    # Compute bin means for plotting
    mbt = np.stack([np.mean(k) for k in bin_t] )
    mbz = np.stack([np.mean(k) for k in bin_z] )
    mbs = np.stack([np.mean(k) for k in bin_s] )
    mbe = np.stack([np.mean(k) for k in bin_e] )
    mba = np.stack([np.mean(k) for k in bin_a] )
    cmap = plt.get_cmap('viridis')
    colors = cmap(np.linspace(0,1,8))
    cmap = mpl.colors.ListedColormap(colors)
    # EA
    ax = axs[0]
    Mba, Mbe = np.meshgrid(mba,mbe)
    im = ax.pcolormesh(Mbe, Mba, nP_EA, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))
    im.set_linewidth(0.001)

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=a_edges,
        counts_xy=nP_EA,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,200,20))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Angular distance $d_{\mathrm{A}}$ [°]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ES
    ax = axs[1]
    Mbs, Mbe = np.meshgrid(mbs,mbe)
    im = ax.pcolormesh(Mbe, Mbs, nP_ES, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=s_edges,
        counts_xy=nP_ES,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Soil dissimilarity $d_{\mathrm{S}}$ [m/s]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # EZ
    ax = axs[2]
    Mbz, Mbe = np.meshgrid(mbz,mbe)
    im = ax.pcolormesh(Mbe, Mbz, nP_EZ, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=z_edges,
        counts_xy=nP_EZ,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Basin dissimilarity $d_{\mathrm{Z1.0}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ET
    ax = axs[3]
    Mbt, Mbe = np.meshgrid(mbt,mbe)
    im = ax.pcolormesh(Mbe, Mbt, nP_ET, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=t_edges,
        counts_xy=nP_ET,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,110,10))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Topographic dissimilarity $d_{\mathrm{TRI}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')

    #colorbar
    cbar = fig.colorbar(im, ax=ax, orientation="vertical",
            ticks=[3, 10, 30, 100, 300, 1000, 3000],
            label='Number of station pairs', drawedges=True)
    cbar.outline.set_linewidth(0.4)
    cbar.outline.set_color('black')
    cbar.dividers.set_linewidth(0.4)
    cbar.dividers.set_color('black')
    cbar.ax.minorticks_off()
    cbar.ax.set_yticklabels(['3', '10','30','100','300','1000', '3000'])
    for a in axs:
        a.grid(True, linewidth=0.3, color='black')
    plt.show()

## 3.6 Computation pairs per period: South Island Group

In [ ]:
# Specify bins
bin_t = [(i, i+10) for i in np.arange(0,100,10)]
bin_z = [(i, i+50) for i in np.arange(0,500,50)]
bin_s = [(i, i+50) for i in np.arange(0,500,50)]
bin_a = [(i, i+20) for i in np.arange(0,180,20)]
bin_e = [(i, i+5) for i in np.arange(0,40,5)]

# define edges
e_edges = np.array([b[0] for b in bin_e] + [bin_e[-1][1]])
a_edges = np.array([b[0] for b in bin_a] + [bin_a[-1][1]])
s_edges = np.array([b[0] for b in bin_s] + [bin_s[-1][1]])
z_edges = np.array([b[0] for b in bin_z] + [bin_z[-1][1]])
t_edges = np.array([b[0] for b in bin_t] + [bin_t[-1][1]])

# outlien threshold
threshold = 30

for ip in [6,11]:
    # Loop over all events
    nP_ET = np.zeros((len(bin_e), len(bin_t)))
    nP_EZ = np.zeros((len(bin_e), len(bin_z)))
    nP_ES = np.zeros((len(bin_e), len(bin_s)))
    nP_EA = np.zeros((len(bin_e), len(bin_a)))
    for _, group in gm_im_table_50_sgrp[ip].groupby('evid'):
        Xtest = group[['epi_dist', 'epi_azimuth', 'Vs30','Z1.0','TRI']].values
        nP_ET += get_npairs_euclT(Xtest, bin_t, bin_e).astype(int)
        nP_EZ += get_npairs_euclZ(Xtest, bin_z, bin_e).astype(int)
        nP_ES += get_npairs_euclS(Xtest, bin_s, bin_e).astype(int)
        nP_EA += get_npairs_angeucl(Xtest, bin_a, bin_e).astype(int)

    # plot
    fig, axs = plt.subplots(1,4, figsize = (12,3), dpi = 200, constrained_layout = True)
    # Compute bin means for plotting
    mbt = np.stack([np.mean(k) for k in bin_t] )
    mbz = np.stack([np.mean(k) for k in bin_z] )
    mbs = np.stack([np.mean(k) for k in bin_s] )
    mbe = np.stack([np.mean(k) for k in bin_e] )
    mba = np.stack([np.mean(k) for k in bin_a] )
    cmap = plt.get_cmap('viridis')
    colors = cmap(np.linspace(0,1,8))
    cmap = mpl.colors.ListedColormap(colors)
    # EA
    ax = axs[0]
    Mba, Mbe = np.meshgrid(mba,mbe)
    im = ax.pcolormesh(Mbe, Mba, nP_EA, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))
    im.set_linewidth(0.001)

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=a_edges,
        counts_xy=nP_EA,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,200,20))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Angular distance $d_{\mathrm{A}}$ [°]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ES
    ax = axs[1]
    Mbs, Mbe = np.meshgrid(mbs,mbe)
    im = ax.pcolormesh(Mbe, Mbs, nP_ES, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=s_edges,
        counts_xy=nP_ES,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Soil dissimilarity $d_{\mathrm{S}}$ [m/s]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # EZ
    ax = axs[2]
    Mbz, Mbe = np.meshgrid(mbz,mbe)
    im = ax.pcolormesh(Mbe, Mbz, nP_EZ, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=z_edges,
        counts_xy=nP_EZ,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,550,50))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Basin dissimilarity $d_{\mathrm{Z1.0}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')
    # ET
    ax = axs[3]
    Mbt, Mbe = np.meshgrid(mbt,mbe)
    im = ax.pcolormesh(Mbe, Mbt, nP_ET, shading='nearest', alpha=0.75,
            cmap = cmap, norm=mpl.colors.LogNorm(vmin=1, vmax=10000))

    outline_threshold_cells(
        ax,
        x_edges=e_edges,
        y_edges=t_edges,
        counts_xy=nP_ET,
        threshold=threshold,
        color="red",
        linewidth=2.0
    )

    ax.set_yticks(np.arange(0,110,10))
    ax.set_xticks(np.arange(0,45,5))
    ax.set_ylabel(r'Topographic dissimilarity $d_{\mathrm{TRI}}$ [m]')
    ax.set_xlabel(r'Euclidean distance $d_{\mathrm{E}}$ [km]')

    #colorbar
    cbar = fig.colorbar(im, ax=ax, orientation="vertical",
            ticks=[3, 10, 30, 100, 300, 1000, 3000],
            label='Number of station pairs', drawedges=True)
    cbar.outline.set_linewidth(0.4)
    cbar.outline.set_color('black')
    cbar.dividers.set_linewidth(0.4)
    cbar.dividers.set_color('black')
    cbar.ax.minorticks_off()
    cbar.ax.set_yticklabels(['3', '10','30','100','300','1000', '3000'])
    for a in axs:
        a.grid(True, linewidth=0.3, color='black')
    plt.show()